# 📚 Web Scraping Pipeline: Books Data

## Objective
Scrape books data (title, price, rating, availability) from a public website and store it in a structured format.

## Use Cases
- Web scraping for analytics
- Price monitoring
- Market analysis

## Tech Stack
- Python
- requests
- BeautifulSoup
- pandas
- datetime

In [ ]:
# import the important libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime

In [2]:
# fetch HTML content using url
url = 'https://books.toscrape.com/'
response = requests.get(url)
response

<Response [200]>

In [3]:
# parse HTML using BeautifulSoup
soup = BeautifulSoup(response.text, 'html.parser')
print(soup.prettify())

<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!-->
<html class="no-js" lang="en-us">
 <!--<![endif]-->
 <head>
  <title>
   All products | Books to Scrape - Sandbox
  </title>
  <meta content="text/html; charset=utf-8" http-equiv="content-type"/>
  <meta content="24th Jun 2016 09:29" name="created"/>
  <meta content="" name="description"/>
  <meta content="width=device-width" name="viewport"/>
  <meta content="NOARCHIVE,NOCACHE" name="robots"/>
  <!-- Le HTML5 shim, for IE6-8 support of HTML elements -->
  <!--[if lt IE 9]>
        <script src="//html5shim.googlecode.com/svn/trunk/html5.js"></script>
        <![endif]-->
  <link href="static/oscar/favicon.ico" rel="shortcut icon"/>
  <link href="static/oscar/css/styles.css" rel="stylesheet" type="tex

In [4]:
# Total number of pages
no_of_pages = int(soup.select('.pager li.current')[0].text.strip().split()[-1])
print(f'There are total {no_of_pages} pages.')

There are total 50 pages.


In [5]:
# Extracting data for all pages

data_list = []
base_url = 'https://books.toscrape.com/catalogue/page-{}.html'
for page in range(1, no_of_pages + 1):
    url = base_url.format(page)
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')

    for i in soup.select('ol.row li'):
        data_dict = {}
        data_dict['name'] = i.select('h3')[-1].text.strip()
        data_dict['price'] = i.select('.product_price p.price_color')[0].text.strip().replace('Â£', '')
        data_dict['rating'] = i.select('p.star-rating')[0].get('class')[-1]
        data_dict['stock_status'] = i.select('p.instock.availability')[0].text.strip()
        data_dict['timestamp'] = datetime.now()
        data_list.append(data_dict)
        
data_list

[{'name': 'A Light in the ...',
  'price': '51.77',
  'rating': 'Three',
  'stock_status': 'In stock',
  'timestamp': datetime.datetime(2026, 5, 4, 21, 43, 36, 740876)},
 {'name': 'Tipping the Velvet',
  'price': '53.74',
  'rating': 'One',
  'stock_status': 'In stock',
  'timestamp': datetime.datetime(2026, 5, 4, 21, 43, 36, 741312)},
 {'name': 'Soumission',
  'price': '50.10',
  'rating': 'One',
  'stock_status': 'In stock',
  'timestamp': datetime.datetime(2026, 5, 4, 21, 43, 36, 741710)},
 {'name': 'Sharp Objects',
  'price': '47.82',
  'rating': 'Four',
  'stock_status': 'In stock',
  'timestamp': datetime.datetime(2026, 5, 4, 21, 43, 36, 742102)},
 {'name': 'Sapiens: A Brief History ...',
  'price': '54.23',
  'rating': 'Five',
  'stock_status': 'In stock',
  'timestamp': datetime.datetime(2026, 5, 4, 21, 43, 36, 742500)},
 {'name': 'The Requiem Red',
  'price': '22.65',
  'rating': 'One',
  'stock_status': 'In stock',
  'timestamp': datetime.datetime(2026, 5, 4, 21, 43, 36, 7428

In [6]:
# Converting the extracted data into pandas DataFrame
df = pd.DataFrame(data_list)
df

,name,price,rating,stock_status,timestamp
0,A Light in the ...,51.77,Three,In stock,2026-05-04 21:43:36.740876
1,Tipping the Velvet,53.74,One,In stock,2026-05-04 21:43:36.741312
2,Soumission,50.10,One,In stock,2026-05-04 21:43:36.741710
3,Sharp Objects,47.82,Four,In stock,2026-05-04 21:43:36.742102
4,Sapiens: A Brief History ...,54.23,Five,In stock,2026-05-04 21:43:36.742500
...,...,...,...,...,...
995,Alice in Wonderland (Alice's ...,55.53,One,In stock,2026-05-04 21:48:28.014379
996,"Ajin: Demi-Human, Volume 1 ...",57.06,Four,In stock,2026-05-04 21:48:28.014758
997,A Spy's Devotion (The ...,16.97,Five,In stock,2026-05-04 21:48:28.015137
998,1st to Die (Women's ...,53.98,One,In stock,2026-05-04 21:48:28.015522


In [7]:
# Shape of the data
print(f'There are {df.shape[0]} rows and {df.shape[1]} columns in the data.')

There are 1000 rows and 5 columns in the data.


In [8]:
# Data type of each column
df.dtypes

name                       str
price                      str
rating                     str
stock_status               str
timestamp       datetime64[us]
dtype: object

In [9]:
# convert the data type of 'Price' from str to float
df['price'] = df['price'].astype('float')
df.dtypes

name                       str
price                  float64
rating                     str
stock_status               str
timestamp       datetime64[us]
dtype: object

In [10]:
# Let's map the ratings

df["rating"] = df["rating"].map({'One':1,
                                 'Two':2,
                                 'Three':3,
                                 'Four':4,
                                 'Five': 5})
df.head()

,name,price,rating,stock_status,timestamp
0,A Light in the ...,51.77,3,In stock,2026-05-04 21:43:36.740876
1,Tipping the Velvet,53.74,1,In stock,2026-05-04 21:43:36.741312
2,Soumission,50.10,1,In stock,2026-05-04 21:43:36.741710
3,Sharp Objects,47.82,4,In stock,2026-05-04 21:43:36.742102
4,Sapiens: A Brief History ...,54.23,5,In stock,2026-05-04 21:43:36.742500


In [11]:
# info of the data
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   name          1000 non-null   str           
 1   price         1000 non-null   float64       
 2   rating        1000 non-null   int64         
 3   stock_status  1000 non-null   str           
 4   timestamp     1000 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(1), int64(1), str(2)
memory usage: 39.2 KB


In [12]:
# Statistical summary of the data
df.describe()

,price,rating,timestamp
count,1000.00000,1000.000000,1000
mean,35.07035,2.923000,2026-05-04 21:46:09.411329
min,10.00000,1.000000,2026-05-04 21:43:36.740876
25%,22.10750,2.000000,2026-05-04 21:44:44.334128
50%,35.98000,3.000000,2026-05-04 21:46:32.632915
75%,47.45750,4.000000,2026-05-04 21:47:24.763912
max,59.99000,5.000000,2026-05-04 21:48:28.015889
std,14.44669,1.434967,NaN


In [13]:
# export this data to csv
df.to_csv("books_to_scrape.csv", index=False)

In [14]:
# no. of unique books
print(f'There are {df['name'].nunique()} unique books.')

There are 992 unique books.


In [15]:
# Top 10 Expensive books
df.sort_values(by = 'price', ascending = False).head(10)

,name,price,rating,stock_status,timestamp
648,The Perfect Play (Play ...,59.99,3,In stock,2026-05-04 21:47:12.314588
617,Last One Home (New ...,59.98,3,In stock,2026-05-04 21:47:08.960278
860,Civilization and Its Discontents,59.95,2,In stock,2026-05-04 21:47:52.645683
560,The Barefoot Contessa Cookbook,59.92,5,In stock,2026-05-04 21:46:56.696376
366,The Diary of a ...,59.90,3,In stock,2026-05-04 21:45:29.066956
657,The Bone Hunters (Lexy ...,59.71,3,In stock,2026-05-04 21:47:12.317770
133,Thomas Jefferson and the ...,59.64,1,In stock,2026-05-04 21:44:01.886580
387,Boar Island (Anna Pigeon ...,59.48,3,In stock,2026-05-04 21:45:31.621904
549,The Man Who Mistook ...,59.45,4,In stock,2026-05-04 21:46:48.728861
393,The Improbability of Love,59.45,1,In stock,2026-05-04 21:45:31.624370


In [16]:
# Top 10 Cheapest books
df.sort_values(by = 'price', ascending = True).head(10)

,name,price,rating,stock_status,timestamp
638,An Abundance of Katherines,10.00,5,In stock,2026-05-04 21:47:10.638510
501,The Origin of Species,10.01,4,In stock,2026-05-04 21:46:42.504771
716,The Tipping Point: How ...,10.02,2,In stock,2026-05-04 21:47:20.479191
84,Patience,10.16,3,In stock,2026-05-04 21:43:57.529876
302,Greek Mythic History,10.23,5,In stock,2026-05-04 21:45:01.833836
558,The Fellowship of the ...,10.27,2,In stock,2026-05-04 21:46:48.732267
479,History of Beauty,10.29,4,In stock,2026-05-04 21:46:14.499324
242,The Lucifer Effect: Understanding ...,10.40,1,In stock,2026-05-04 21:44:44.331290
434,NaNo What Now? Finding ...,10.41,4,In stock,2026-05-04 21:45:44.839545
274,Pet Sematary,10.56,3,In stock,2026-05-04 21:44:57.355690


In [17]:
# Rating Analysis
print('Average Price by Rating')
df.groupby('rating')['price'].mean()

Average Price by Rating


rating
1    34.561195
2    34.810918
3    34.692020
4    36.093296
5    35.374490
Name: price, dtype: float64

In [18]:
print('No. of books by rating')
df.rating.value_counts().sort_values(ascending = False)

No. of books by rating


rating
1    226
3    203
5    196
2    196
4    179
Name: count, dtype: int64

In [19]:
# stock status
df['stock_status'].value_counts()

stock_status
In stock    1000
Name: count, dtype: int64

**All 1000 books are in stock.**

## Key Learnings

- Handling pagination in web scraping
- Parsing structured HTML using BeautifulSoup
- Data cleaning and transformation using pandas